In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

df = pd.read_csv("../data/raw/fraud_oracle.csv")

print("Dataset shape:", df.shape)

Dataset shape: (15420, 33)


In [2]:
target = "FraudFound_P"

X = df.drop(columns=[target, "PolicyNumber"])
y = df[target]

print("Feature matrix:", X.shape)
print("Target:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Feature matrix: (15420, 31)
Target: (15420,)

Target distribution:
FraudFound_P
0    14497
1      923
Name: count, dtype: int64


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("\nTraining fraud percentage:")
print(y_train.mean() * 100)

print("\nTesting fraud percentage:")
print(y_test.mean() * 100)

Training records: 12336
Testing records: 3084

Training fraud percentage:
5.982490272373541

Testing fraud percentage:
5.998702983138781


In [4]:
X_train = X_train.drop(columns=["RepNumber"]).copy()
X_test = X_test.drop(columns=["RepNumber"]).copy()

X_train["Age"] = X_train["Age"].replace(0, np.nan)
X_test["Age"] = X_test["Age"].replace(0, np.nan)

print("Training missing ages:", X_train["Age"].isna().sum())
print("Testing missing ages:", X_test["Age"].isna().sum())

Training missing ages: 251
Testing missing ages: 69


In [5]:
categorical_features = X_train.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

categorical_features += ["Year", "DriverRating"]

numerical_features = [
    col for col in X_train.select_dtypes(include="number").columns
    if col not in categorical_features
]

print("Categorical features:", categorical_features)
print("\nNumerical features:", numerical_features)

Categorical features: ['Month', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'Sex', 'MaritalStatus', 'Fault', 'PolicyType', 'VehicleCategory', 'VehiclePrice', 'Days_Policy_Accident', 'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'PoliceReportFiled', 'WitnessPresent', 'AgentType', 'NumberOfSuppliments', 'AddressChange_Claim', 'NumberOfCars', 'BasePolicy', 'Year', 'DriverRating']

Numerical features: ['WeekOfMonth', 'WeekOfMonthClaimed', 'Age', 'Deductible']


In [6]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully")

Preprocessing pipeline created successfully


In [7]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

print("\nMissing values after preprocessing:")
print("Training:", np.isnan(X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed).sum())
print("Testing:", np.isnan(X_test_processed.toarray() if hasattr(X_test_processed, "toarray") else X_test_processed).sum())

Training shape: (12336, 151)
Testing shape: (3084, 151)

Missing values after preprocessing:
Training: 0
Testing: 0


In [8]:
median_age = X_train["Age"].median()

print("Training median age:", median_age)
print("Training ages requiring imputation:", X_train["Age"].isna().sum())
print("Testing ages requiring imputation:", X_test["Age"].isna().sum())

Training median age: 39.0
Training ages requiring imputation: 251
Testing ages requiring imputation: 69
